In [ ]:
# Modelin her müşteri için churn olasılığını hesaplar ve belirtilen eşik değerinin
# üzerindeki müşterileri belirler. Müşteri bilgilerini koruyarak churn olasılığını
# sonuçlara ekler ve müşterileri en yüksek olasılıktan en düşüğe doğru sıralar.
def predict(model, X, customer_info, threshold=0.5):

    probs = model.predict_proba(X)[:, 1]

    result = customer_info.copy()

    result["churn_probability"] = probs

    result = result[result["churn_probability"] >= threshold]

    result = result.sort_values("churn_probability", ascending=False)

    return result

In [ ]:
# Modeli eğitim verisi üzerinde eğitir ve test verisi üzerinden performansını takip eder.
# Early stopping kullanarak modelin 50 tur boyunca gelişme göstermemesi durumunda
# eğitimi sonlandırır ve eğitim sürecine ait sonuçları döndürür.
def train(model, X_train, X_test, y_train, y_test):
    history = model.fit(
              X_train,
              y_train,
              eval_set=(X_test, y_test),
              early_stopping_rounds=50
            )

    return history

In [ ]:
import os


# Eğitilmiş modeli belirtilen dosya yoluna kaydeder.
# Modelin kaydedileceği klasör mevcut değilse gerekli klasörleri oluşturur.
# Kayıt sırasında bir hata oluşursa işlemin neden başarısız olduğunu belirten
# bir RuntimeError oluşturur.
def save_model(model, path):

    os.makedirs(os.path.dirname(path), exist_ok=True)

    try:

        model.save_model(path)

    except Exception as e:

        raise RuntimeError(f"Could not save model to {path}: {e}")

In [ ]:
from catboost import CatBoostClassifier


# CatBoost sınıflandırma modelini verilen hiperparametrelerle oluşturur.
# İterasyon sayısı, ağaç derinliği ve öğrenme oranı gibi model ayarlarını
# belirler; kategorik özellikleri destekler ve sınıf dengesizliğini
# otomatik olarak dikkate alacak şekilde yapılandırılır.
def build_model(iterations, depth, learning_rate, cat_features):

    model = CatBoostClassifier(

        iterations=iterations,

        depth=depth,

        learning_rate=learning_rate,

        cat_features=cat_features,

        auto_class_weights="Balanced",

        eval_metric="AUC",

        random_state=42,

        verbose=False,

    )

    return model

In [ ]:
import os

from catboost import CatBoostClassifier


# Belirtilen dosya yolundaki eğitilmiş CatBoost modelini yükler.
# Öncelikle model dosyasının mevcut olup olmadığını kontrol eder,
# ardından modeli dosyadan yükler. Yükleme sırasında bir hata oluşursa
# işlemin neden başarısız olduğunu belirten bir RuntimeError oluşturur.
def load_model(path):

    if not os.path.exists(path):

        raise FileNotFoundError(f"Model file not found: {path}")

    model = CatBoostClassifier()

    try:

        model.load_model(path)

    except Exception as e:

        raise RuntimeError(f"Could not load model from {path}: {e}")

    return model

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# Test verisi üzerinde modelin tahmin performansını değerlendirir.
# Tahmin olasılıklarını belirtilen eşik değerine göre sınıflandırır ve
# accuracy, precision, recall, F1, confusion matrix ve ROC-AUC gibi
# farklı performans metriklerini hesaplayarak sonuçları bir sözlükte toplar.
def evaluate_model(model, X_test, y_test, threshold: float = 0.5) -> dict:
    probs = model.predict_proba(X_test)[:, 1]
    y_pred = (probs >= threshold).astype(int)

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "classification_report": classification_report(
            y_test, y_pred, output_dict=True, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_test, probs),
    }

In [ ]:
import shap


# Verilen model için SHAP açıklayıcısı oluşturur.
# Tree tabanlı modeller için uygun olan TreeExplainer kullanılarak
# modelin tahminlerinin hangi özelliklerden etkilendiğini analiz etmeye hazırlar.
def create_explainer(model):
    return shap.TreeExplainer(model)


# Her müşteri için SHAP değerlerini hesaplayarak tahmini en fazla etkileyen
# özellikleri belirler. Özelliklerin etkilerini mutlak değerlerine göre sıralar
# ve yalnızca belirtilen top_n sayıdaki özelliğin açıklamasını döndürür.
def explain_customers(explainer, customer_data, top_n):
    shap_values = explainer.shap_values(customer_data)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    feature_names = list(customer_data.columns)
    explanations = []

    for i in range(len(customer_data)):
        customer_shap = shap_values[i]

        feature_impacts = [
            {"feature": feature, "impact": float(value)}
            for feature, value in zip(feature_names, customer_shap)
        ]

        feature_impacts.sort(key=lambda x: abs(x["impact"]), reverse=True)
        explanations.append(feature_impacts[:top_n])

    return explanations

In [ ]:
"""Veri kalitesi doğrulama işlemlerini gerçekleştirir.
Beklenen sütunların mevcut olup olmadığını, veri setinin boş olup olmadığını
ve kritik seviyedeki eksik veri oranını kontrol eder. Ayrıca veri setindeki
tekrarlanan kayıtları tespit ederek doğrulama sonucunu özetler ve loglar.
"""

import logging

logger = logging.getLogger(__name__)


class DataValidationError(Exception):
    """Veri seti, pipeline'ın devam etmesi için gerekli minimum kalite koşullarını karşılamadığında kullanılır."""


# Veri setinin beklenen sütunları içerip içermediğini ve eksik veri oranının
# kabul edilen sınırlar içinde olup olmadığını kontrol eder. Ayrıca boş veri
# setini ve tekrar eden kayıtları tespit ederek doğrulama sonuçlarını özetler.
def validate(df, required_columns: list[str], max_null_ratio: float = 0.05) -> dict:
    missing_columns = [c for c in required_columns if c not in df.columns]

    if missing_columns:
        raise DataValidationError(
            f"expected columns are missing: {missing_columns}. existing columns: {list(df.columns)}"
        )

    if df.empty:
        raise DataValidationError("DataFrame is empty.")

    null_counts = df[required_columns].isnull().sum()
    total_cells = len(df) * len(required_columns)
    null_ratio = float(null_counts.sum()) / total_cells if total_cells else 0.0

    if null_ratio > max_null_ratio:
        raise DataValidationError(
            f"Null rate is High: %{null_ratio * 100:.2f} "
            f"(Accepted rate: %{max_null_ratio * 100:.2f})"
        )

    duplicate_count = int(df.duplicated().sum())

    summary = {
        "shape": df.shape,
        "columns": list(df.columns),
        "null_counts": null_counts.to_dict(),
        "duplicate_count": duplicate_count,
    }

    if duplicate_count > 0:
        logger.warning("Duplicated value found in dataset.", duplicate_count)

    logger.info("Data Validation is Finished %s", summary)
    return summary

In [ ]:
from sklearn.model_selection import train_test_split


# Veri setinden öğrenciye ait bilgi sütunlarını ve hedef değişkeni ayırarak
# modelde kullanılacak özellikleri (X), hedef değişkeni (y) ve kategorik
# sütunların veri setindeki konumlarını belirler.
def target_split(df, STUDENT_INFO, TARGET_FEATURE, CAT_COLS):
    X = df.drop(columns=STUDENT_INFO)
    X_last = X.drop(columns=TARGET_FEATURE)
    cat_features = [X_last.columns.get_loc(c) for c in CAT_COLS]
    y = df[TARGET_FEATURE].squeeze()

    return X_last, y, cat_features


# Veriyi eğitim ve test kümelerine ayırır. Verinin %80'i eğitim,
# %20'si test için kullanılır ve stratify ile hedef değişkenin sınıf
# dağılımının her iki kümede de korunmasını sağlar.
def train_test_splits(X_last, y, cat_features):
    X_train, X_test, y_train, y_test = train_test_split(
        X_last,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    return X_train, X_test, y_train, y_test, cat_features


# Veri setini modelleme için hazırlar. Özellik ve hedef değişkenleri ayırır,
# veriyi eğitim ve test kümelerine böler ve kategorik sütunları modelin
# kullanabileceği string formatına dönüştürür.
def preprocess(df, STUDENT_INFO, TARGET_FEATURE, CAT_COLS):
    X_last, y, cat_features = target_split(df, STUDENT_INFO, TARGET_FEATURE, CAT_COLS)
    X_train, X_test, y_train, y_test, cat_features = train_test_splits(X_last, y, cat_features)

    for col in CAT_COLS:
        X_train[col] = X_train[col].astype(str)
        X_test[col] = X_test[col].astype(str)

    return X_train, X_test, y_train, y_test, cat_features


# Günlük işlemde kullanılmayacak kimlik/bilgi sütunlarını ayırır.
# Orijinal öğrenci bilgilerini customer_info içinde tutarken,
# modelleme veya sonraki işlemlerde kullanılacak verileri x_daily olarak döndürür.
def daily_process(df, id_columns):
    customer_info = df[list(id_columns)]
    x_daily = df.drop(columns=id_columns)
    return customer_info, x_daily

In [ ]:
import os

import pandas as pd


# Verilen dosya yolunun geçerli olup olmadığını kontrol eder.
# Dosya mevcutsa CSV verisini pandas DataFrame formatında yükler;
# bulunamazsa işlemin devam etmesini engellemek için hata oluşturur.
def data_loader(PATH):

    if not os.path.exists(PATH):

        raise FileNotFoundError(f"Data file not found: {PATH}")

    return pd.read_csv(PATH)